# Performance Analysis

### **Identificação do Trabalho**
*   **Unidade Curricular**: Ciência de Dados em Larga Escala 
*   **Instituição**: Faculdade de Ciências da Universidade do Porto
*   **Ano Letivo**: 2025/2026

### **Trabalho realizado por:**
*   **João Levandeira** 
*   **Nuno Antunes**

---

# NOTEBOOK 1
## Estudo Comparativo de Desempenho e Escalabilidade de Bibliotecas de Big Data (CPU Benchmarking)
Este notebook apresenta um estudo comparativo rigoroso e empírico de processamento de dados em larga escala na nuvem, focado na medição de tempos de execução, consumo de recursos e limites de escalabilidade vertical de cinco ecossistemas em Python: **Pandas**, **Dask**, **PySpark (Koalas)**, **Modin** e **Joblib**.
O processamento é realizado diretamente sobre a infraestrutura do **GCP**, extraindo dados do **GCS** e executando as tarefas num cluster distribuído do **GCP Dataproc**.

### 1. Objetivos do Benchmark
Alinhado com as diretrizes da proposta de projeto CDLE, este estudo visa atingir quatro objetivos fundamentais:
1. **Identificar Performance Bottlenecks**: Avaliar onde cada biblioteca falha fisicamente ao escalar o volume de dados.
2. **Identificar Diferenças Sintáticas**: Confrontar o paradigma de execução imediata com o paradigma de execução lazy, mapeando a complexidade do código.
3. **Mapear Casos de Uso Ideais**: Compreender qual a ferramenta mais adequada para cada tipo de operação (Leitura, Contagem, Frequência, Agrupamento e Filtragem) e para cada tipo de dataset.
4. **Avaliar a Compatibilidade**: Avaliar a integração e suporte que estas bibliotecas fornecem a ferramentas de processamento clássico em Python (Pandas, NumPy e Scikit-Learn).


---

### 2. Arquitetura Cloud e Integração GCP

#### 2.1. Configuração de Desenvolvimento (Single-Node)
* **Estrutura**: 1 Master Node da família `n4-highmem-4`.
* **Especificações por Nó**:
  * **Processamento**: 4 vCPUs.
  * **Memória RAM**: 32 GB.
  * **Armazenamento**: 50 GB SSD persistente local.
* **Propósito**: Destinado à escrita inicial de código, depuração, testes rápidos de sintaxe e verificação de concorrência local antes da submissão ao cluster.

---

#### 2.2. Configuração Distribuída de Alta Performance (Multi-Node)
Para simular um ambiente corporativo real e processar os conjuntos de dados de média e grande escala utilizou-se um cluster gerido via **GCP Dataproc** com a seguinte topologia de rede:

* **Master Node**:
  * 1 instância `n4-highmem-4` (4 vCPUs, 32 GB RAM).
* **Worker Nodes**:
  * 2 instâncias `n4-standard-4` dedicadas unicamente à execução dos cálculos.
* **Recursos Dedicados de Computação Distribuída**:
  * **Processamento total**: 4 vCPUs dedicadas.
  * **Memória RAM total**: 16 GB RAM

Ambas estruturas utilizam ligação à internet.

---

#### 2.3. Estrutura usada no RapidAI
Para o benchmarking de GPU utilizando **RapidsAI** (`cuDF`, `Dask-cuDF`, `cuML`):

* **Estrutura**: 1 Máquina Virtual do **Google Colabs**.
* **Especificações Gráficas**:
  * **Acelerador Físico**: GPU NVIDIA Tesla T4.
  * **Memória Gráfica (VRAM)**: 16 GB.
  * **Tecnologia**: Aceleradores baseados em arquitetura de paralelização massiva de núcleos CUDA e Tensor Cores dedicados a operações matemáticas rápidas em matrizes colunares.

---

Todos os dados de entrada e saída estão integrados diretamente com os serviços de armazenamento em nuvem do Google Cloud atráves da leitura do GCP, `gs://`.

---

## Guia do projeto

Para visualizar o trabalho de acordo com a estrutura desejada, os quatro notebooks do projeto seguem a seguinte ordem:

### 1. Experimento #1: Estudo Comparativo de CPU 
* **Notebook Correspondente**: `1_benchmarks.ipynb`.
* **Conteúdo**: Leitura e execução de benchmarks nas 5 frameworks em CPU (Pandas, Dask, PySpark/Koalas, Modin, Joblib) sob os 3 datasets, com o relatório de comparação de tempos final e gráfico consolidado de escalabilidade.

### 2. Experimento #2: Aceleração por GPU e Profiling de CPU
* **Notebooks Correspondentes**: 
  1. `2_profiling.ipynb`: Profiling determinístico de funções internas de CPU via `cProfile` para identificar os bootlenecks.
  2. `3_rapidai_benchmark.ipynb`: Execução acelerada por hardware em GPU utilizando RapidsAI (`cuDF`, `Dask-cuDF` e `Modin-Rapid`) e comparação contra os clusters CPU.

### 3. Previsão: Pipeline de Machine Learning
* **Notebook Correspondente**: `4_ml_pipeline.ipynb`
* **Conteúdo**: Pipeline preditiva completa e sem data leakage contem a modelação por Regressão (fare_amount contínuo) e Classificação (tarifa discretizada por quantis), resultados de precisão, F1-score e importâncias de atributos.

---

In [ ]:
import sys
!{sys.executable} -m pip install -q "modin[ray]" matplotlib

In [ ]:
# --- PATCH GLOBAL DE VISUALIZAÇÃO DE TABELAS ---
import pandas as pd
if not hasattr(pd.Index, '_format_flat'):
    pd.Index._format_flat = lambda self, *args, **kwargs: [str(x) for x in self]

try:
    import pyspark.pandas as ps
    if not hasattr(ps.Index, '_format_flat'):
        ps.Index._format_flat = lambda self, *args, **kwargs: [str(x) for x in self]
except Exception:
    pass
# -----------------------------------------------

import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

---

### Parametrização da Escalabilidade
Para simular a escalabilidade e o gerir os créditos, a leitura de dados permite selecionar três escalas de execução interativas:
1. **Escala Pequena**: Amostra representativa inicial de **cerca de 123.000 linhas**.
2. **Escala Média**: **cerca de 2.460.000 linhas**.
3. **Escala Grande**: Dataset completo com mais de **9.000.000 linhas**.


O bloco abaixo corre uma verificação rápida no GCS usando o utilitário `gsutil`. Se os ficheiros `_small` e `_large` não existirem, inicializa uma sessão Spark temporária para gerá-los de forma distribuída diretamente no Google Cloud Storage, desligando-a em seguida para libertar 100% da RAM para o benchmark.

In [ ]:
# --- GERADOR DE DATASETS DE ESCALONAMENTO NO GCS ---
import subprocess
from pyspark.sql import SparkSession

# Definição dos caminhos no GCS (Staging local do cluster)
base_dir = "gs://dataproc-staging-europe-southwest1-348488791616-f80l4tzf/notebooks/jupyter/"
medium_uri = base_dir + "yellow_tripdata_2022-01.parquet"
small_uri = base_dir + "yellow_tripdata_2022_small.parquet"
large_uri = base_dir + "yellow_tripdata_2022_large.parquet"

def gcs_file_exists(uri):
    try:
        # Executa gsutil ls para verificar se o ficheiro/pasta existe no GCS
        result = subprocess.run(["gsutil", "ls", uri], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return result.returncode == 0
    except Exception:
        return False

exists_small = gcs_file_exists(small_uri)
exists_large = gcs_file_exists(large_uri)

if not exists_small or not exists_large:
    spark_temp = SparkSession.builder \
        .appName("CDLE-Dataset-Generator") \
        .getOrCreate()
    
    # Ler o dataset padrão (Média Escala)
    df_base = spark_temp.read.parquet(medium_uri)
    
    if not exists_small:
        df_small = df_base.sample(withReplacement=False, fraction=0.05, seed=42)
        df_small.write.mode("overwrite").parquet(small_uri)
        
    if not exists_large:
        # Unimos o dataset 3 vezes para simular 9M de registos
        df_feb = spark_temp.read.parquet(base_dir + "yellow_tripdata_2022-02.parquet")
        df_mar = spark_temp.read.parquet(base_dir + "yellow_tripdata_2022-03.parquet")
        df_large = df_base.union(df_feb).union(df_mar)
        df_large.write.mode("overwrite").parquet(large_uri)
        
    spark_temp.stop()

Menu de navegação, que nos permite escolher o dataset que queremos utilizar.

In [ ]:
print(" 1 - Escala Pequena (~123k linhas)")
print(" 2 - Escala Média (~2.46M linhas)")
print(" 3 - Escala Grande (~9M linhas)")

while True:
    try:
        op = input("Introduza a opcao desejada (1, 2 ou 3): ").strip()
        ESCALA_DATASET = int(op)
        if ESCALA_DATASET in [1, 2, 3]:
            break
        else:
            print("Opcao invalida. Por favor, introduza 1, 2 ou 3.")
    except ValueError:
        print("Entrada invalida. Por favor, introduza um numero.")

base_dir = "gs://dataproc-staging-europe-southwest1-348488791616-f80l4tzf/notebooks/jupyter/"
caminho = {
    1: base_dir + "yellow_tripdata_2022_small.parquet",
    2: base_dir + "yellow_tripdata_2022-01.parquet",
    3: base_dir + "yellow_tripdata_2022_large.parquet"
}

file_path = caminho[ESCALA_DATASET]

pandas_results = {}
dask_results = {}
pyspark_results = {}
modin_results = {}
joblib_results = {}

---
## Parte 1: Pandas (O Padrão Eager e Single-Node)

O **Pandas** é a biblioteca de referência absoluta para ciência de dados em Python. No entanto, foi desenhado sob premissas específicas que determinam a sua viabilidade em Big Data:

### Características Principais:
* **Execução Eager:** Cada linha de código é compilada e executada de imediato. Qualquer operação gera imediatamente um novo objeto concreto na memória RAM.
* **Arquitetura Single-Node e Single-Threaded:** O Pandas opera estritamente num único computador e, devido ao Global Interpreter Lock do Python, a maior parte das suas operações corre numa única thread.
* **Memory Footprint:** Como regra geral, o Pandas requer entre **2 a 3 vezes mais memória RAM** do que o tamanho do dataset em disco para realizar operações de cópia, ordenação e alinhamento de índices. Isto torna-o altamente suscetível a falhas de *Out of Memory* com datasets médios.

### 1.1 Read Data (Leitura do Parquet)

In [ ]:
start = time.time()
df_pd = pd.read_parquet(file_path)
pandas_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura Pandas: {pandas_results['1. Read']:.3f} s")
display(df_pd.head(5))

### 1.2 Count (Contagem de Linhas)

In [ ]:
start = time.time()
total_rows_pd = len(df_pd)
pandas_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem Pandas: {pandas_results['2. Count']:.3f} s (Total de viagens: {total_rows_pd})")

### 1.3 Value Counts (Frequências)
Contar quantas viagens foram processadas por cada fornecedor (`VendorID`).

In [ ]:
start = time.time()
vc_pd = df_pd['VendorID'].value_counts()
pandas_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts Pandas: {pandas_results['3. Value Counts']:.3f} s")
display(vc_pd.to_frame())

### 1.4 GroupBy (Agrupamento e Agregação)
Agrupar pelo tipo de pagamento (`payment_type`) e calcular a tarifa média.

In [ ]:
start = time.time()
gb_pd = df_pd.groupby('payment_type')['fare_amount'].mean()
pandas_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy Pandas: {pandas_results['4. GroupBy']:.3f} s")
display(gb_pd.to_frame())

### 1.5 Add Column (Mutação de Dados)
Criar uma nova coluna `Total_Calculated` adicionando a tarifa, gorjeta e portagens.

In [ ]:
start = time.time()
df_pd['Total_Calculated'] = df_pd['fare_amount'] + df_pd['tip_amount'] + df_pd['tolls_amount']
pandas_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de Coluna Pandas: {pandas_results['5. Add Column']:.3f} s")
display(df_pd[['fare_amount', 'tip_amount', 'tolls_amount', 'Total_Calculated']].head(2))

### 1.6 Filter (Filtragem)
Encontrar todas as viagens que custaram estritamente mais de 10 dólares.

In [ ]:
start = time.time()
filtered_pd = df_pd[df_pd['fare_amount'] > 10]
pandas_results['6. Filter'] = time.time() - start

print(f"Tempo de Filtragem Pandas: {pandas_results['6. Filter']:.3f} s (Total filtrado: {len(filtered_pd)})")

Libertar memória do Pandas antes de iniciar o Dask

In [ ]:
import gc
del df_pd, filtered_pd
gc.collect()

---
## Parte 2: Dask

O **Dask** é uma biblioteca de computação flexível e paralela que expande o ecossistema científico do Python (Pandas, NumPy, Scikit-Learn) para sistemas distribuídos e processamento que excede a memória RAM disponível.

### Como Funciona o Dask:
* **Arquitetura Particionada:** Um `dask.dataframe` é composto por múltiplos pequenos DataFrames do Pandas, divididos por partições de linhas. O Dask gere a execução de operações sobre cada partição de forma coordenada.
* **Execução Lazy:** Ao contrário do Pandas, o Dask não executa as operações imediatamente. Em vez disso, ele constrói um grafo que mapeia todas as tarefas necessárias. A computação real só é disparada quando invocamos uma ação explícita, como `.compute()`, `.head()` ou `len()`.
* **Otimização de Grafo:** O motor do Dask analisa o grafo para fundir operações, evitando leituras desnecessárias de colunas e poupando memória.
* **Gestão de Memória Estável:** Permite processar datasets muito maiores do que a memória RAM física através do processamento sequencial de partições e descarte automático de dados intermédios (*garbage collection*).

Para garantir estabilidade no nó Master, configurámos o agendador do Dask para modo síncrono/sequencial, garantindo que o processamento de partições não gere picos acumulados de RAM.

### 2.1 Read Data (Leitura do Parquet)

In [ ]:
# Monkey-patch para resolver incompatibilidade entre Dask antigo e Python 3.11+
try:
    import dask.utils
    original_derived_from = dask.utils.derived_from
    def safe_derived_from(*args, **kwargs):
        decorator = original_derived_from(*args, **kwargs)
        def safe_decorator(func):
            try:
                return decorator(func)
            except Exception:
                return func
        return safe_decorator
    dask.utils.derived_from = safe_derived_from
    import dask
    dask.config.set(scheduler='synchronous') # Execução sequencial para evitar OOM no Master node
except Exception as e:
    print(f"Aviso ao aplicar patch do Dask: {e}")

import dask.dataframe as dd

start = time.time()
df_dd = dd.read_parquet(file_path)
dask_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura Dask: {dask_results['1. Read']:.3f} s (Note-se que a avaliação é Lazy)")

### 2.2 Count (Conta as linhas)

In [ ]:
start = time.time()
total_rows_dd = len(df_dd)
dask_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem Dask: {dask_results['2. Count']:.3f} s")

### 2.3 Value Counts (Frequências)
Contar quantas viagens foram processadas por cada fornecedor (`VendorID`).

In [ ]:
start = time.time()
# Temos de forçar o cálculo da distribuição com o .compute()
vc_dd = df_dd['VendorID'].value_counts().compute()
dask_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts Dask: {dask_results['3. Value Counts']:.3f} s")
display(vc_dd.to_frame())

### 2.4 GroupBy (Agrupamento e Agregação)
Agrupar pelo tipo de pagamento (`payment_type`) e calcular a tarifa média.

In [ ]:
start = time.time()
gb_dd = df_dd.groupby('payment_type')['fare_amount'].mean().compute()
dask_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy Dask: {dask_results['4. GroupBy']:.3f} s")
display(gb_dd.to_frame())

### 2.5 Add Column (Mutação de Dados)
Criar uma nova coluna `Total_Calculated` adicionando a tarifa, gorjeta e portagens.

In [ ]:
start = time.time()
df_dd['Total_Calculated'] = df_dd['fare_amount'] + df_dd['tip_amount'] + df_dd['tolls_amount']
preview_dd = df_dd['Total_Calculated'].head() # Usamos o head() para simular o trigger da ação
dask_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de Coluna Dask: {dask_results['5. Add Column']:.3f} s")

### 2.6 Filter (Filtragem)
Encontrar todas as viagens que custaram estritamente mais de 10 dólares.

In [ ]:
start = time.time()
filtered_dd = df_dd[df_dd['fare_amount'] > 10].head()
dask_results['6. Filter'] = time.time() - start

print(f"Tempo de Filtragem Dask: {dask_results['6. Filter']:.3f} s")

Libertar memória do Dask antes de iniciar o PySpark, para não haver crash do clutcher.

In [ ]:
import gc
del df_dd, filtered_dd
gc.collect()

---
## Parte 3: PySpark (Processamento Massivo Distribuído - Spark Engine)

O **PySpark** fornece a interface Python para o motor do Spark, e através da API **Pandas-on-Spark**, antigo Koalas, permite executar código com sintaxe Pandas diretamente sobre a arquitetura do Spark.

### O Motor do PySpark:
* **Execução Distribuída Massiva:** O Spark foi desenhado desde a raiz para funcionar com clusters. O Master node planeia a execução e distribui as partições de dados para múltiplos Worker nodes, que processam a informação em paralelo.
* **Lazy Evaluation:** Tal como o Dask, o Spark é lazy. No entanto, ele possui o otimizador *Catalyst*, que traduz o código em planos de execução lógicos altamente otimizados antes de iniciar qualquer computação.
* **Resistência e Tolerância a Falhas:** Os dados são mantidos em estruturas resilientes. Se um worker falhar a meio da computação, o Spark reconstrói automaticamente a partição perdida a partir do grafo de linhagem.
* **API Pandas-on-Spark (`pyspark.pandas`):** Permite o uso de dados familiarizados com o Pandas e escalam os seus códigos para petabytes de dados mantendo a lógica de manipulação.

### 3.1 Read Data (Leitura do Parquet)

In [ ]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'
from pyspark.sql import SparkSession
import pyspark.pandas as ps

# Inicializar o Spark (no Dataproc, ele utiliza o YARN por defeito)
spark = SparkSession.builder \
    .appName('CDLE-Benchmark') \
    .config('spark.sql.ansi.enabled', 'false') \
    .getOrCreate()

start = time.time()

sdf = spark.read.parquet(file_path)
for col_name, col_type in sdf.dtypes:
    if 'timestamp' in col_type or 'ntz' in col_type.lower():
        sdf = sdf.withColumn(col_name, sdf[col_name].cast('timestamp'))

df_ps = sdf.to_pandas_on_spark()
pyspark_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura PySpark: {pyspark_results['1. Read']:.3f} s")
display(df_ps.head(2))

### 3.2 Count (Conta as linhas)

In [ ]:
start = time.time()
total_rows_ps = len(df_ps)
pyspark_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem PySpark: {pyspark_results['2. Count']:.3f} s")

### 3.3 Value Counts (Frequências)
Contar quantas viagens foram processadas por cada fornecedor (`VendorID`).

In [ ]:
start = time.time()
vc_ps = df_ps['VendorID'].value_counts()
pyspark_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts PySpark: {pyspark_results['3. Value Counts']:.3f} s")
display(vc_ps.to_frame())

### 3.4 GroupBy (Agrupamento e Agregação)
Agrupar pelo tipo de pagamento (`payment_type`) e calcular a tarifa média.

In [ ]:
start = time.time()
gb_ps = df_ps.groupby('payment_type')['fare_amount'].mean()
pyspark_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy PySpark: {pyspark_results['4. GroupBy']:.3f} s")
display(gb_ps.to_frame())

### 3.5 Add Column (Mutação de Dados)
Criar uma nova coluna `Total_Calculated` adicionando a tarifa, gorjeta e portagens.

In [ ]:
start = time.time()
df_ps['Total_Calculated'] = df_ps['fare_amount'] + df_ps['tip_amount'] + df_ps['tolls_amount']
pyspark_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de Coluna PySpark: {pyspark_results['5. Add Column']:.3f} s")

### 3.6 Filter (Filtragem)
Encontrar todas as viagens que custaram estritamente mais de 10 dólares.

In [ ]:
start = time.time()
filtered_ps = df_ps[df_ps['fare_amount'] > 10].head()
pyspark_results['6. Filter'] = time.time() - start

print(f"Tempo de Filtragem PySpark: {pyspark_results['6. Filter']:.3f} s")

# Parar SparkSession e libertar memória para o gráfico final
import gc
spark.stop()
del df_ps, filtered_ps
gc.collect()

---
## Parte 4: Modin

O modin é uma biblioteca de código aberto em Python projetada para acelarar e dimensionar fluxos de trabalho da biblioteca Pandas. Esta biblioteca funciona como um substituto direto do Pandas, sendo asim a sintaxe semelhante à do pandas, mas por baixo do "capô" ele distribui as operações pelas CPUs da máquina, usando do Dask ou Ray.

### 4.1 Read Data

In [ ]:
# Benchmark Modin
import os
os.environ["MODIN_ENGINE"] = "python"

# Patch de compatibilidade Modin e Pandas 2.1.4
import sys
import pandas as pd
try:
    import pandas.core.arrays.arrow
except ImportError:
    import types
    sys.modules['pandas.core.arrays.arrow'] = types.ModuleType('pandas.core.arrays.arrow')
    import pandas.core.arrays.arrow

if not hasattr(pandas.core.arrays.arrow, 'ListAccessor'):
    class DummyListAccessor: pass
    pandas.core.arrays.arrow.ListAccessor = DummyListAccessor

if not hasattr(pandas.core.arrays.arrow, 'StructAccessor'):
    class DummyStructAccessor: pass
    pandas.core.arrays.arrow.StructAccessor = DummyStructAccessor


import time
import modin.pandas as mpd
from IPython.display import display

# Dicionário para armazenar os tempos de execução do Modin
modin_results = {}

start = time.time()
df_mod = mpd.read_parquet(file_path)
modin_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura Modin: {modin_results['1. Read']:.3f} s")
display(df_mod.head(2))


### 4.2 Count (Conta as linhas)

In [ ]:
start = time.time()
total_rows_mod = len(df_mod)
modin_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem Modin: {modin_results['2. Count']:.3f}s (Total de viagens: {total_rows_mod})")

### 4.3 Value Counts (Frequências)
Contar quantas viagens foram processadas por cada fornecedor (`VendorID`).

In [ ]:
start = time.time()
val_mod = df_mod['VendorID'].value_counts()
modin_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts com Modin: {modin_results['3. Value Counts']:.3f}s")
display(val_mod.to_frame())

### 4.4 GroupBy (Agrupamento e Agregação)
Agrupar pelo tipo de pagamento (`payment_type`) e calcular a tarifa média.

In [ ]:
start = time.time()
gb_mod = df_mod.groupby('payment_type')['fare_amount'].mean()
modin_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy com Modin: {modin_results['4. GroupBy']:.3f}s")
display(gb_mod.to_frame())

### 4.5 Add Column (Mutação de Dados)
Criar uma nova coluna `Total_Calculated` adicionando a tarifa, gorjeta e portagens.

In [ ]:
start = time.time()
df_mod['Total_Calculated'] = df_mod['fare_amount'] + df_mod['tip_amount'] + df_mod['tolls_amount']
modin_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de coluna com Modin:{modin_results['5. Add Column']:.3f}s")
display(df_mod[['fare_amount', 'tip_amount', 'tolls_amount', 'Total_Calculated']].head(2))

### 4.6 Filter (Filtragem)
Encontrar todas as viagens que custaram estritamente mais de 10 dólares.

In [ ]:
start = time.time()
filtered_mod = df_mod[df_mod['fare_amount'] > 10].head()
modin_results['6. Filter'] = time.time() - start

print(f"Tempo de filtragem com Modin: {modin_results['6. Filter']:.3f}s")

Para limpar a memória ativa do modin, para evitar o crash dos clusters

In [ ]:
import gc
del df_mod, filtered_mod
gc.collect()

---
## Parte 5: Joblib

O Joblib é uma biblioteca de paralelização genérica baseada em multiprocessing. Como o Joblib não tem uma estrutura de dados própria, o padrão seria dividir o dataframe em fatias/chunks e processá-las em paralelo no Joblib, e por fim combinar os resultados parciais.

In [ ]:
import time
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from IPython.display import display

joblib_results = {}

### 5.1 Read Data

In [ ]:
start = time.time()
df_pd_job = pd.read_parquet(file_path)
joblib_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura com Joblib: {joblib_results['1. Read']:.3f} s")
display(df_pd_job.head(2))

### 5.2 Count

In [ ]:
start = time.time()
chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs = 4)(delayed(len)(chunk) for chunk in chunks)
total_rows_job = sum(results)
joblib_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem com Joblib: {joblib_results['2. Count']:.3f} s")

### 5.3 Value Counts (Frequências)
Contar quantas viagens foram processadas por cada fornecedor (`VendorID`).

In [ ]:
def val_chunk(chunk):
    return chunk['VendorID'].value_counts()

start = time.time()
chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs = 4)(delayed(val_chunk)(chunk) for chunk in chunks)
vc_job = pd.concat(results).groupby(level=0).sum()
joblib_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts com Joblib: {joblib_results['3. Value Counts']:.3f} s")
display(vc_job.to_frame())

### 5.4 GroupBy
Para calcular a média global corretamente em paralelo, precisamos de obter a soma e o count parciais

In [ ]:
def gb_chunk(chunk):
    return chunk.groupby('payment_type')['fare_amount'].agg(['sum', 'count'])

chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs=4)(delayed(gb_chunk)(chunk) for chunk in chunks)
combined = pd.concat(results).groupby('payment_type').sum()
gb_job = combined['sum'] / combined['count']
joblib_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy Joblib: {joblib_results['4. GroupBy']:.3f} s")
display(gb_job.to_frame())


### 5.5 Add Column (Mutação de Dados)
Criar uma nova coluna `Total_Calculated` adicionando a tarifa, gorjeta e portagens.

In [ ]:
def add_col_chunk(chunk):
    chunk['Total_Calculated'] = chunk['fare_amount'] + chunk['tip_amount'] + chunk['tolls_amount']
    return chunk

start = time.time()
chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs=4)(delayed(add_col_chunk)(chunk) for chunk in chunks)

df_pd_job = pd.concat(results)
joblib_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de Coluna com Joblib: {joblib_results['5. Add Column']:.3f} s")
display(df_pd_job[['fare_amount', 'tip_amount', 'tolls_amount', 'Total_Calculated']].head(2))

### 5.6 Filter (Filtragem)
Encontrar todas as viagens que custaram estritamente mais de 10 dólares.

In [ ]:
def filter_chunk(chunk):
    return chunk[chunk['fare_amount'] > 10]

start = time.time()
chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs=4)(delayed(filter_chunk)(chunk) for chunk in chunks)
filtered_job = pd.concat(results)
joblib_results['6. Filter'] = time.time() - start

print(f"Tempo de Filtragem com Joblib: {joblib_results['6. Filter']:.3f} s")

Para limpar a memória ativa do joblib, para não causar um crash dos clusters.

In [ ]:
import gc
del df_pd_job, filtered_job
gc.collect()

---
## Parte 6: Análise Comparativa e Resumo Gráfico

Reunimos nesta secção final os tempos de execução guardados em todas as secções anteriores. O código abaixo analisa dinamicamente quais as bibliotecas que correram com sucesso e constrói uma tabela resumo consolidada e um gráfico de barras estilizado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import os

# Definir a escala do dataset (garantir que existe uma variavel padrao se correr diretamente)
if 'ESCALA_DATASET' not in locals():
    ESCALA_DATASET = 2

escala_nome = {1: 'Pequena', 2: 'Média', 3: 'Grande'}.get(ESCALA_DATASET, 'Média')

active_runs = {}

try:
    if 'pandas_results' in locals() and pandas_results:
        active_runs['Pandas (s)'] = pandas_results
except NameError: pass

try:
    if 'dask_results' in locals() and dask_results:
        active_runs['Dask (s)'] = dask_results
except NameError: pass

try:
    if 'pyspark_results' in locals() and pyspark_results:
        active_runs['PySpark (s)'] = pyspark_results
except NameError: pass

try:
    if 'modin_results' in locals() and modin_results:
        active_runs['Modin (s)'] = modin_results
except NameError: pass

try:
    if 'joblib_results' in locals() and joblib_results:
        active_runs['Joblib (s)'] = joblib_results
except NameError: pass

if active_runs:
    results_df = pd.DataFrame(active_runs)
    
    print('='*60)
    print(f'   RESUMO COMPARATIVO DE EXECUCAO - ESCALA {escala_nome.upper()}   ')
    print('='*60)
    display(results_df.round(3))
    
    # Forcar uso do estilo padrao se o seaborn v0.8 nao estiver disponivel
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Plot de barras comparativo nativo em Matplotlib para evitar o bug de import do backend do Pandas no GCP
    import numpy as np
    x = np.arange(len(results_df))
    cols = results_df.columns
    width = 0.8 / max(1, len(cols))
    cmap = plt.colormaps.get_cmap('plasma')
    
    for i, col in enumerate(cols):
        color_val = cmap(i / max(1, len(cols) - 1)) if len(cols) > 1 else cmap(0.3)
        offset = (i - (len(cols) - 1) / 2.0) * width
        ax.bar(x + offset, results_df[col], width, label=col, color=color_val, edgecolor='black', alpha=0.9)
        
    ax.set_xticks(x)
    ax.set_xticklabels(results_df.index)
    
    plt.title(f'Estudo Comparativo de Performance - Escala {escala_nome} (Menos Segundos = Mais Rapido)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Operacoes Benchmarked', fontsize=12, fontweight='bold', labelpad=10)
    plt.ylabel('Tempo de Execucao (segundos)', fontsize=12, fontweight='bold', labelpad=10)
    
    plt.xticks(rotation=30, ha='right', fontsize=11)
    plt.yticks(fontsize=11)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6, color='#cccccc')
    ax.xaxis.grid(False)
    
    plt.legend(title='Arquiteturas / Frameworks Ativos', title_fontsize='11', fontsize='10', 
               loc='upper right', frameon=True, shadow=True, facecolor='white')
    
    plt.tight_layout()
    plt.show()

# Análise de Performance e Discussão de Resultados

Para compreender a eficiência de cada ecossistema ao lidar com Big Data em Python, foi executado um conjunto sistemático de testes comparativos no GCP. Os testes cobrem três escalas volumétricas de dados reais retirados do dataset de viagens de táxi de Nova Iorque:
* **Escala Pequena**: cerca de123.000 linhas.
* **Escala Média**: cerca de2.460.000 linhas.
* **Escala Grande**: cerca de 9.000.000 linhas.

---

## 1. Tabelas Consolidadas de Tempos de Execução (Segundos)

Abaixo encontram-se registados os tempos reais de execução de cada biblioteca ao realizar operações comuns de manipulação de dados:

### Tabela A: Escala Pequena
| Operação | Pandas (s) | Dask (s) | PySpark (s) | Modin (s) | Joblib (s) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **1. Read** | 0.292 | 0.249 | 5.152 | 0.255 | 0.202 |
| **2. Count** | 0.000 | 0.157 | 1.186 | 0.000 | 0.685 |
| **3. Value Counts** | 0.002 | 0.079 | 0.077 | 0.035 | 0.141 |
| **4. GroupBy** | 0.003 | 0.084 | 0.067 | 0.036 | 0.290 |
| **5. Add Column** | 0.002 | 0.129 | 0.051 | 0.066 | 0.165 |
| **6. Filter** | 0.006 | 0.125 | 0.159 | 0.032 | 0.147 |

### Tabela B: Escala Média
| Operação | Pandas (s) | Dask (s) | PySpark (s) | Modin (s) | Joblib (s) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **1. Read** | 0.749 | 0.494 | 7.942 | 0.853 | 0.577 |
| **2. Count** | 0.000 | 0.549 | 0.839 | 0.001 | 1.003 |
| **3. Value Counts** | 0.015 | 0.160 | 0.056 | 0.266 | 0.576 |
| **4. GroupBy** | 0.039 | 0.289 | 0.059 | 0.492 | 1.124 |
| **5. Add Column** | 0.011 | 1.065 | 0.049 | 0.991 | 1.420 |
| **6. Filter** | 0.113 | 1.148 | 0.165 | 0.602 | 1.065 |

### Tabela C: Escala Grande 
| Operação | Pandas (s) | Dask (s) | PySpark (s) | Modin (s) | Joblib (s) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **1. Read** | 1.524 | 0.246 | 4.198 | 2.254 | 1.354 |
| **2. Count** | 0.000 | 0.511 | 0.812 | 0.001 | 2.148 |
| **3. Value Counts** | 0.030 | 0.418 | 0.068 | 1.028 | 1.737 |
| **4. GroupBy** | 0.119 | 0.857 | 0.065 | 1.898 | 3.505 |
| **5. Add Column** | 0.041 | 3.840 | 0.036 | 4.462 | 5.231 |
| **6. Filter** | 0.439 | 4.145 | 0.137 | 3.530 | 4.211 |

---

## 2. Discussão e Análise Crítica dos Resultados

### A. PySpark — Otimização Física Plana
* **Efeito JVM e Custo Fixo**: Em escala pequena, o PySpark apresenta um tempo de carregamento inicial substancial ($5.152\text{ s}$), reflexo da inicialização da `SparkSession` e da sobrecarga da JVM local. 
* **Aquecimento de Nós**: Na escala grande, o tempo de leitura desceu para $4.198\text{ s}$. Isto comprova o comportamento de aquecimento do Spark. Uma vez instanciado e aquecido o cluster, a gestão de dados torna-se mais ágil.
* **Escalabilidade Imbatível**: A principal virtude do PySpark é a sua curva de tempo estável. Operações de agregação mantiveram-se quase inalteradas em todas as três escalas ($0.067\text{ s} \rightarrow 0.059\text{ s} \rightarrow 0.065\text{ s}$). O *Catalyst Optimizer* reordena e planeia o processamento em partições de forma tão otimizada que o aumento de volume dos datasets não afetou a performance.

### B. Pandas — Eficiência Local com Riscos de Memória
* **Desempenho Vetorizado**: O Pandas sequencial demonstrou tempos de execução excelentes nas escalas pequena e média, mantendo tempos lineares mesmo na escala de 9 milhões de linhas (leitura a $1.524\text{ s}$, `GroupBy` a $0.119\text{ s}$ e filtragem a $0.439\text{ s}$).
* **Explicação Técnica**: Como os dados são lidos em bloco e residem por completo dentro da memória RAM da instância GCP (cerca de $150\text{ MB}$ para a escala média e menos de $1\text{ GB}$ para a escala grande), o Pandas tira partido da vetorização puramente compilada em C/C++ do motor subjacente. Sem custos de comunicação inter-processos (IPC) ou divisão de partições pela CPU, a execução local sequencial é imbatível a volumes médios.
* **Limitações Físicas**: Apesar da velocidade local, o Pandas não é escalável horizontalmente. Se o tamanho do dataset exceder a RAM física disponível da máquina, o sistema gera um erro catastrófico por esgotamento de memória, situação em que falha por completo, ao contrário do Dask ou do PySpark.

### C. Dask e Modin — A Sobrecarga da Orquestração de Tarefas locais
* **Lazy Evaluation**: A leitura inicial do Dask é quase instantânea ($0.246\text{ s}$ na escala grande) porque a biblioteca executa em formato *lazy*. O Dask apenas lê os metadados do arquivo Parquet e constrói o plano lógico de tarefas, adiando a computação real.
* **Custo de Gestão de Tarefas**: Ao executar transformações e filtros, o agendador local do Dask e do Modin gerou tempos de processamento elevados (Dask `Filter`: $4.145\text{ s}$; Modin `Filter`: $3.530\text{ s}$). A orquestração, sincronização e a divisão local de micro-tarefas pelas várias partições em memória acabam por gerar um custo de gestão que ultrapassa o ganho de paralelização numa escala em que a execução vetorizada linear do Pandas, ainda cabe confortavelmente no espaço de memória RAM da máquina.

### D. Joblib — O Custo da Serialização Inter-Processos (IPC)
* **Pior Desempenho à Escala**: O Joblib registou os tempos mais lentos na escala grande (ex: $5.231\text{ s}$ na adição de colunas e $3.505\text{ s}$ em agrupamentos).
* **Explicação Técnica**: O Joblib realiza processamento paralelo manual dividindo fisicamente o DataFrame do Pandas em fatias através de `np.array_split()`. Para correr o código em paralelo, o Joblib é forçado a serializar estes grandes DataFrames e a enviá-los através de buffers do sistema operativo para processos separados na CPU. Este custo pesado de serialização e posterior reagregação dos pedaços de volta na memória anula por completo os benefícios da paralelização de múltiplos núcleos.

---

## 3. Diretrizes Metodológicas de Escolha para Engenharia de Dados
Com base nos testes experimentais efetuados nas diferentes volumetrias, definem-se as seguintes recomendações:
1. **Pandas**: A melhor escolha para exploração rápida de dados locais e prototipagem rápida, desde que o volume caiba na RAM.
2. **PySpark**: A única escolha adequada para produção e ambientes distribuídos reais de Big Data, apresentando escalabilidade plana e horizontal.
3. **Dask / Modin**: Alternativas úteis para processar dados locais que excedam pontualmente os limites de RAM da máquina sem necessidade de migrar para a arquitetura Java/Scala do PySpark.

---

## Desafios Técnicos e Resoluções Práticas

Ao longo do desenvolvimento deste projeto em nuvem, enfrentou-se uma série de obstáculos de engenharia de software e infraestrutura. A documentação sincera de como estes problemas foram superados constitui um elemento de elevado valor técnico para o relatório:

1. **Restrição de Ligação Externa (Firewall das VMs no Dataproc)**:
   * *O Problema*: Descarregar os dados brutos diretamente para as máquinas virtuais do cluster Dataproc.
   * *A Resolução*: Implementou-se o streaming direto de dados via biblioteca `gcsfs` no Pandas.

2. **Bloqueio Completo por Paralelização Aninhada (Thread Deadlock)**:
   * *O Problema*: Ao configurar `n_jobs=-1` no `GridSearchCV` e também nos estimadores internos, o sistema gerava processos paralelos que tentavam criar outros processos paralelos. No GCP, isto causava colisão de recursos, congelando a execução do JupyterLab de forma indefinida.
   * *A Resolução*: Desativou-se o paralelismo nos estimadores individuais, mantendo o paralelismo ativado apenas no nível superior do `GridSearchCV`.

3. **Erros de Esgotamento de Memória**:
   * *O Problema*: Executar processos de validação cruzada sobre o maior dataset na máquina virtual excedia os limites de memória da instância, dando crash do kernel.
   * *A Resolução*: Criou-se uma Célula de Configuração Global introduzindo uma amostragem estatística controlada (`SAMPLE_SIZE = 100000`). Isto reduziu a pegada de memória para limites seguros e rápidos, e otimizou-se a grelha de parâmetros para focar apenas nas combinações mais determinantes, reduzindo o tempo de treino de horas para escassos segundos.

4. **Erro de Compatibilidade de Timestamp NTZ no PySpark Pandas API (Koalas)**:
   * *O Problema*: Os ficheiros Parquet de 2022 gravados pela TLC usam o tipo `TimestampNTZType` do Spark. Ao carregar estes ficheiros no PySpark Pandas API com `ps.read_parquet()`, o frame interno falhava devido à incapacidade de mapear este tipo para NumPy em ambientes de Dataproc antigos.
   * *A Resolução*: Substituiu-se a leitura direta por um pipeline híbrido. O ficheiro Parquet é primeiro carregado como um DataFrame padrão do Spark. Em seguida, todas as colunas do tipo timestamp são convertidas programaticamente para o tipo de timestamp padrão do Spark, que é totalmente compatível. Por fim, o DataFrame é convertido nativamente para o Pandas-on-Spark com, resolvendo o bug de forma imediata.


5. **Arquitetura Modular CPU/GPU**:
   * *O Problema*: O RapidsAI e as bibliotecas aceleradas de GPU exigem hardware físico NVIDIA dedicado. Tentar colocar o código de GPU dentro do mesmo notebook principal de CPU, fazia com que fossem gastos muitos créditos.
   * *A Resolução*: Fizemos um notebook novo apenas para este tópico, mantendo os testes de CPU focados e limpos no GCP Dataproc e isolando a execução da GPU em neste notebook que é executado no Google Colab com GPU T4.